# 01 — Volatility Dynamics: expected volatility, surprise, buildup and decay

### Research purpose
This branch asks what the **volatility process itself** looks like before returning to MOM/MR/VS/Bridge hypotheses. Module 01 does **not** fit a latent state model and does not test direction.

It constructs causal 30m/60m physical-volatility forecasts from the existing HAR+clock M1 machinery, then separates:

- **expected high volatility** — future RV is high and the causal forecast was already high;
- **volatility shock candidate** — future RV is high and the realized outcome exceeds its forecast by an unusually large amount;
- **forecast-high miss** — the forecast was high but realized volatility did not enter the high-RV tail.

The event labels and `surprise_*` variables are **retrospective diagnostics only**. They are never used in the origin-time forecast.

### Holdout protection
The frozen Volatility Regression confirmation begins with expiry sessions **2026-10-09**. This notebook therefore defaults to development data ending **2026-10-08** and refuses to cross that boundary unless an explicit override is supplied. Running this notebook normally does not inspect the protected confirmation outcomes.

### Why exact-clock normalization appears here
Flow, liquidity, IV and activity diagnostics are standardized relative to the same exact five-minute UTC clock slot. This prevents ordinary deterministic time-of-day structure from being mistaken for a volatility-event precursor.


In [ ]:
from pathlib import Path
import json, os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 260)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found from current working directory.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_dynamics as vd

START = os.getenv('VD_START', '2026-09-22')
END = os.getenv('VD_END', '2026-10-08')
ALLOW_PROTECTED = os.getenv('VD_ALLOW_CONFIRMATION_CONTAMINATION', '0') == '1'
MIN_TRAIN_SESSIONS = int(os.getenv('VD_MIN_TRAIN_SESSIONS', '7'))
EVENT_Q = float(os.getenv('VD_EVENT_Q', '0.90'))
CLUSTER_GAP_MIN = int(os.getenv('VD_CLUSTER_GAP_MIN', '30'))
ASSET_FILTER = [x.strip().upper() for x in os.getenv('VD_ASSETS', '').split(',') if x.strip()]
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '01'
OUT.mkdir(parents=True, exist_ok=True)

if pd.Timestamp(END) >= pd.Timestamp(vd.PROTECTED_CONFIRMATION_START) and not ALLOW_PROTECTED:
    raise RuntimeError(
        f'VD_END={END} enters the protected Volatility Regression confirmation window '
        f'starting {vd.PROTECTED_CONFIRMATION_START}. Leave VD_END at 2026-10-08 or set '
        'VD_ALLOW_CONFIRMATION_CONTAMINATION=1 only after Ivan explicitly releases that holdout.'
    )
print('volatility_dynamics', vd.VOLATILITY_DYNAMICS_VERSION)
print('selftest:', vd.selftest())
print(f'development window {START} -> {END} | min train sessions {MIN_TRAIN_SESSIONS} | event q {EVENT_Q}')


## 1. Load only the data needed for the volatility-state panel

The physical forecast uses index returns + deterministic clock only. Additional flow, L2, basis/OI and option fields are diagnostics; absent fields remain missing rather than being imputed.


In [ ]:
required = {
    'state_id','state_time','underlying','return_5m','futures_index_price','is_weekend',
    'in_settlement_window','option_atm_quote_ok','option_iv_atm_forward'
}
diagnostic_sources = {src for src, _ in vd.DIAGNOSTIC_SPECS.values()} | {
    'book_twa_bid_depth_20','book_twa_ask_depth_20'
}

con = duckdb.connect(str(DB_PATH), read_only=True)
available = set(con.execute('DESCRIBE time_series_feature_state').df()['column_name'])
missing = sorted(required - available)
if missing:
    raise RuntimeError(f'time_series_feature_state lacks required columns: {missing}')
cols = sorted(required | (diagnostic_sources & available))
query_start = (pd.Timestamp(START, tz='UTC') - pd.Timedelta(days=1) + pd.Timedelta(hours=8))
query_end = pd.Timestamp(END, tz='UTC') + pd.Timedelta(hours=8)
asset_sql = '' if not ASSET_FILTER else ' AND underlying IN (' + ','.join(repr(x) for x in ASSET_FILTER) + ')'
df = con.execute(
    f"SELECT {', '.join(cols)} FROM time_series_feature_state "
    f"WHERE state_time >= TIMESTAMPTZ '{query_start.isoformat()}' "
    f"AND state_time <= TIMESTAMPTZ '{query_end.isoformat()}'{asset_sql} "
    "ORDER BY underlying, state_time"
).df()
con.close()
df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
assets = sorted(df['underlying'].dropna().unique())
print(f'rows {len(df):,} | assets {assets} | {df.state_time.min()} -> {df.state_time.max()}')
print('optional diagnostics present:', sorted(diagnostic_sources & available))


## 2. Causal expanding HAR forecasts

For each expiry session, M1 is re-estimated using **strictly earlier sessions**. This is an exploratory real-time-style forecast panel, not a frozen confirmation model. IV is excluded from M1 so that future work can compare physical expectations with option-implied information without circularity.


In [ ]:
panels, fit_tables = [], []
diag_z_names = None
for asset in assets:
    g = df[df.underlying == asset].sort_values('state_time').reset_index(drop=True)
    try:
        p, fits = vd.expanding_m1_state_panel(g, min_train_sessions=MIN_TRAIN_SESSIONS)
        p, zcols = vd.merge_diagnostics(p, g, min_clock_n=8)
        panels.append(p)
        fit_tables.append(fits)
        diag_z_names = zcols if diag_z_names is None else diag_z_names
        okfits = int((fits.status == 'ok').sum()) if len(fits) else 0
        print(f'{asset}: rows {len(p):,} | successful session×horizon fits {okfits}')
    except Exception as exc:
        print(f'{asset}: SKIPPED {type(exc).__name__}: {exc}')

if not panels:
    raise RuntimeError('No asset produced a volatility state panel.')
state = pd.concat(panels, ignore_index=True)
fits = pd.concat(fit_tables, ignore_index=True) if fit_tables else pd.DataFrame()
quality = vd.forecast_quality(state)
display(quality.round(4))


## 3. Separate forecastable high volatility from surprise

Within each asset, the 90th percentiles of causal forecast level, realized 30m variance rate and positive forecast surprise define descriptive candidate rows. Adjacent overlapping candidates are then **declustered** so one volatility episode is not counted six times merely because 30m targets overlap every five minutes.

These quantile labels are development-data descriptions, not pre-registered live states.


In [ ]:
state, thresholds = vd.label_volatility_events(state, q=EVENT_Q, horizon='30m')
events = vd.decluster_event_origins(state, cluster_gap_min=CLUSTER_GAP_MIN, horizon='30m')
counts = vd.event_counts(events)
display(thresholds.round(4))
display(counts)


## 4. What is different *before* each kind of event?

Every market diagnostic below is expressed as a robust z-score relative to the same exact five-minute UTC slot. We report both the event-origin value and the trailing 15-minute mean. The aim is to discover candidate precursors—not to call them causal or predictive yet.


In [ ]:
diag_z = [c for c in state.columns if c.endswith('_clock_z')]
feature_summary = vd.event_feature_summary(state, events, diag_z, pre_bars=3)
if len(feature_summary):
    show = feature_summary[(feature_summary.view == 'pre_window') & (feature_summary.n_events >= 2)].copy()
    display(show.sort_values(['underlying','event_type','median'], ascending=[True,True,False]).head(120).round(3))
else:
    print('No event-feature summaries available.')


## 5. Buildup and decay paths

Paths run from 60 minutes before to 120 minutes after each declustered event origin, without crossing the expiry-session boundary. The core series are the causal forecast level, forecast acceleration, forecast term slope and backward realized-volatility level; clock-normalized market diagnostics are added where available.


In [ ]:
path_vars = [
    'log_forecast_rate_eps_30m','log_forecast_rate_eps_60m',
    'log_back_rate_30m','log_back_rate_1h','log_back_rate_4h',
    'forecast_accel_30m_15m','forecast_accel_30m_30m',
    'forecast_term_slope_60v30','back_term_slope_30v4h'
] + diag_z
paths = vd.event_path_summary(state, events, path_vars, pre_min=60, post_min=120)
print(f'aggregated path rows: {len(paths):,}')

# One compact in-notebook visual: BTC if available, otherwise first asset/event pair.
if len(paths):
    core = paths[paths.variable == 'log_forecast_rate_eps_30m'].copy()
    if len(core):
        chosen_asset = 'BTC' if 'BTC' in set(core.underlying) else core.underlying.iloc[0]
        fig, ax = plt.subplots(figsize=(9, 4))
        for kind, gg in core[core.underlying == chosen_asset].groupby('event_type'):
            ax.plot(gg.rel_min, gg['median'], marker='o', markersize=2, label=kind)
        ax.axvline(0, linewidth=1)
        ax.set_title(f'{chosen_asset}: causal 30m volatility forecast around event origins')
        ax.set_xlabel('Minutes relative to event origin')
        ax.set_ylabel('Median log forecast variance per bar')
        ax.legend()
        plt.show()


## 6. Persist only compact review artifacts

No large state-panel dump or plot archive is committed by default. The notebook can always rebuild the panel from DuckDB and the versioned helper.


In [ ]:
quality.to_csv(OUT / '01_forecast_quality.csv', index=False)
fits.to_csv(OUT / '01_fit_status.csv', index=False)
thresholds.to_csv(OUT / '01_event_thresholds.csv', index=False)
events.to_csv(OUT / '01_event_origins.csv', index=False)
counts.to_csv(OUT / '01_event_counts.csv', index=False)
feature_summary.to_csv(OUT / '01_event_feature_summary.csv', index=False)
paths.to_csv(OUT / '01_event_paths.csv', index=False)

meta = {
    'version': vd.VOLATILITY_DYNAMICS_VERSION,
    'research_question': 'Expected volatility vs volatility surprise; buildup and decay around declustered 30m events',
    'database_relation': 'time_series_feature_state',
    'database_read_only': True,
    'start_session': START,
    'end_session': END,
    'protected_confirmation_start': vd.PROTECTED_CONFIRMATION_START,
    'protected_override_used': ALLOW_PROTECTED,
    'min_train_sessions': MIN_TRAIN_SESSIONS,
    'event_quantile': EVENT_Q,
    'cluster_gap_minutes': CLUSTER_GAP_MIN,
    'assets_processed': sorted(state.underlying.unique().tolist()),
    'n_state_rows': int(len(state)),
    'n_events': int(len(events)),
    'diagnostic_clock_z_features': diag_z,
    'forecast': 'expanding previous-session-only HAR+clock M1; IV excluded',
    'event_labels': 'retrospective descriptive only; never forecast inputs',
    'output_policy': 'compact summaries only; full state panel recomputed locally',
}
(OUT / '01_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote outputs to', OUT)
display(counts)


## Interpretation boundary

This module can support statements such as:

- high volatility was often forecastable versus genuinely surprising;
- particular flow/liquidity/IV diagnostics tended to build before one event class but not another;
- the causal volatility forecast tended to accelerate or decay before/after events.

It **cannot** establish MOM, MR, Bridge or a discrete Volatility-Shock state. A later module may use the observable volatility environment defined here to test whether displacement/absorption has different forward-return behavior conditional on that environment.
